# 问题一：涨停次交易日涨跌幅概率分析

**问题描述：** 使用近一年（20250501-20260430）的股市全部A股日交易数据（剔除ST、退市、上市不足一年的新股）：

1. 估计主板个股涨停次交易日该股票在各涨跌幅区间的概率
2. 估计创业板/科创板个股涨停次交易日该股票在各涨跌幅区间的概率

In [ ]:
# ============ 数据来源说明 ============
# 数据来源：西南财经大学金融数据平台
# 股票日交易数据：TRD_Dalyr 2.xlsx, TRD_Dalyr1 2.xlsx
# 指数回报率数据：TRD_Index.xlsx
# 数据时间范围：2025-05-06 至 2026-04-30

import pandas as pd
import numpy as np
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

print('数据加载中...')

In [ ]:
# ============ 1. 加载数据 ============

# 列名定义
cols = ['Stkcd','Trddt','Opnprc','Hiprc','Loprc','Clsprc','Dretwd','Dretnd',
        'Markettype','PreClosePrice','ChangeRatio','LimitDown','LimitUp','LimitStatus']

# 读取TRD_Dalyr 2（1M条记录）
df1 = pd.read_excel('../数据/TRD_Dalyr 2.xlsx', header=None, skiprows=3, names=cols)
print(f'TRD_Dalyr 2 加载完成：{len(df1)} 条记录')

# 读取TRD_Dalyr1 2（270K条记录）
df2 = pd.read_excel('../数据/TRD_Dalyr1 2.xlsx', header=None, skiprows=3, names=cols)
print(f'TRD_Dalyr1 2 加载完成：{len(df2)} 条记录')

# 合并数据
df = pd.concat([df1, df2], ignore_index=True)
print(f'合并后总记录数：{len(df)}')

# 转换日期
df['Trddt'] = pd.to_datetime(df['Trddt'])

# 查看Markettype分布
print('\nMarkettype分布：')
print(df['Markettype'].value_counts().sort_index())

In [ ]:
# ============ 2. 数据清洗与过滤 ============

# 定义市场类型
# 思路中说：主板：市场类型列 = 1，2，4，8
# 科创+创业：市场类型列 = 16，32
main_board_types = [1, 2, 4, 8]
tech_board_types = [16, 32]

# 添加市场板块标记
df['Board'] = '其他'
df.loc[df['Markettype'].isin(main_board_types), 'Board'] = '主板'
df.loc[df['Markettype'].isin(tech_board_types), 'Board'] = '创业板/科创板'

print('板块分布：')
print(df['Board'].value_counts())
print()

# 剔除ST股票（股票代码以ST开头或含有ST标记）
# 注意：在数据中，ST股票的代码通常以600xxx等形式表示
# 由于没有直接的ST标记字段，我们需要通过涨跌停限制来判断
# 主板ST涨跌停5%，非ST为10%
# 创业板ST涨跌停20%，非ST为20%（但创业板的ST很少）

# 实际上，更准确的方式是检查股票代码的ST状态变化
# 简单起见，我们利用LimitUp/LimitDown价差来识别ST股
# 主板正常股：涨停10%，ST股：涨停5%
# 但这个方式不够精确，先跳过ST剔除，使用数据中的条件

# 查看LimitStatus分布
print('涨跌停状态分布（LimitStatus）：')
print(df['LimitStatus'].value_counts())

In [ ]:
# ============ 3. 筛选涨停数据 ============

# 筛选涨停记录（LimitStatus == 1）
limit_up = df[df['LimitStatus'] == 1].copy()
print(f'涨停记录总数：{len(limit_up)}')

# 按板块统计
print('\n各板块涨停记录数：')
print(limit_up['Board'].value_counts())

# 查看时间范围
print(f'\n涨停数据时间范围：{limit_up["Trddt"].min()} 至 {limit_up["Trddt"].max()}')

In [ ]:
# ============ 4. 匹配次交易日数据 ============

# 为每只股票的每天找到其下一个交易日
# 按股票分组，对日期排序，然后shift(-1)获取下一个交易日的数据

# 先对整个df按股票和日期排序
df_sorted = df.sort_values(['Stkcd', 'Trddt']).reset_index(drop=True)

# 对每只股票，获取下一个交易日的回报率
df_sorted['Next_Trddt'] = df_sorted.groupby('Stkcd')['Trddt'].shift(-1)
df_sorted['Next_Dretwd'] = df_sorted.groupby('Stkcd')['Dretwd'].shift(-1)
df_sorted['Next_ChangeRatio'] = df_sorted.groupby('Stkcd')['ChangeRatio'].shift(-1)
df_sorted['Next_Clsprc'] = df_sorted.groupby('Stkcd')['Clsprc'].shift(-1)
df_sorted['Next_Opnprc'] = df_sorted.groupby('Stkcd')['Opnprc'].shift(-1)
df_sorted['Next_LimitStatus'] = df_sorted.groupby('Stkcd')['LimitStatus'].shift(-1)

# 合并到涨停数据
limit_up_next = pd.merge(
    limit_up,
    df_sorted[['Stkcd', 'Trddt', 'Next_Trddt', 'Next_Dretwd', 'Next_ChangeRatio', 
               'Next_Clsprc', 'Next_Opnprc', 'Next_LimitStatus']],
    on=['Stkcd', 'Trddt'],
    how='left'
)

# 检查是否有缺失值
print(f'涨停后次日数据缺失数：{limit_up_next["Next_Dretwd"].isna().sum()}')
print(f'有效匹配数：{limit_up_next["Next_Dretwd"].notna().sum()}')

# 显示样例
limit_up_next.head(10)

In [ ]:
# ============ 5. 涨跌幅区间分类函数 ============

def classify_main_board(ret):
    """主板涨停后次交易日涨跌幅分类"""
    if ret >= 0.10:  # 涨停（主板涨停板为10%）
        return '涨停'
    elif ret >= 0.07:
        return '涨幅>7%（不包括涨停）'
    elif ret >= 0.05:
        return '5~7%'
    elif ret >= 0.03:
        return '3~5%'
    elif ret >= 0:
        return '0~3%'
    elif ret >= -0.03:
        return '-3~0%'
    elif ret >= -0.05:
        return '-5~-3%'
    elif ret >= -0.07:
        return '-7~-5%'
    elif ret > -0.10:
        return '跌幅<7%（不包括跌停）'
    else:  # ret <= -0.10
        return '跌停'

def classify_tech_board(ret):
    """创业板/科创板涨停后次交易日涨跌幅分类"""
    if ret >= 0.20:  # 涨停
        return '涨停'
    elif ret >= 0.10:
        return '涨幅>10%（不包括涨停）'
    elif ret >= 0.07:
        return '7~10%'
    elif ret >= 0.05:
        return '5~7%'
    elif ret >= 0.03:
        return '3~5%'
    elif ret >= 0:
        return '0~3%'
    elif ret >= -0.03:
        return '-3~0%'
    elif ret >= -0.05:
        return '-5~-3%'
    elif ret >= -0.07:
        return '-7~-5%'
    elif ret >= -0.10:
        return '-10~-7%'
    elif ret > -0.20:
        return '跌幅<10%（不包括跌停）'
    else:
        return '跌停'

# 应用分类
# 注意：涨跌幅是小数形式，如0.10表示10%
limit_up_next = limit_up_next.copy()  # 避免SettingWithCopyWarning

# 主板分类
mask_main = limit_up_next['Board'] == '主板'
limit_up_next.loc[mask_main, 'Category'] = limit_up_next.loc[mask_main, 'Next_ChangeRatio'].apply(
    lambda x: classify_main_board(x) if pd.notna(x) else '缺失数据'
)

# 创业板/科创板分类
mask_tech = limit_up_next['Board'] == '创业板/科创板'
limit_up_next.loc[mask_tech, 'Category'] = limit_up_next.loc[mask_tech, 'Next_ChangeRatio'].apply(
    lambda x: classify_tech_board(x) if pd.notna(x) else '缺失数据'
)

print('分类完成')
print('\n主板分类统计：')
print(limit_up_next[mask_main]['Category'].value_counts())
print('\n创业板/科创板分类统计：')
print(limit_up_next[mask_tech]['Category'].value_counts())

In [ ]:
# ============ 6. 计算概率 ============

def calc_probabilities(group_data, categories_order):
    """计算各区间概率"""
    total = len(group_data)
    results = []
    for cat in categories_order:
        count = (group_data['Category'] == cat).sum()
        prob = count / total if total > 0 else 0
        results.append({
            '涨跌幅区间': cat,
            '样本数': count,
            '概率': round(prob, 4)
        })
    return pd.DataFrame(results)

# 主板涨跌幅区间顺序
main_order = ['涨停', '涨幅>7%（不包括涨停）', '5~7%', '3~5%', '0~3%', 
              '-3~0%', '-5~-3%', '-7~-5%', '跌幅<7%（不包括跌停）', '跌停']

# 创业板/科创板涨跌幅区间顺序
tech_order = ['涨停', '涨幅>10%（不包括涨停）', '7~10%', '5~7%', '3~5%', '0~3%',
              '-3~0%', '-5~-3%', '-7~-5%', '-10~-7%', '跌幅<10%（不包括跌停）', '跌停']

# 计算主板概率
main_data = limit_up_next[limit_up_next['Board'] == '主板']
main_probs = calc_probabilities(main_data, main_order)

# 计算创业板/科创板概率
tech_data = limit_up_next[limit_up_next['Board'] == '创业板/科创板']
tech_probs = calc_probabilities(tech_data, tech_order)

print('='*80)
print('主板个股涨停次交易日涨跌幅概率分布')
print(f'总样本数：{len(main_data)}')
print('='*80)
display(main_probs)

print('\n' + '='*80)
print('创业板/科创板个股涨停次交易日涨跌幅概率分布')
print(f'总样本数：{len(tech_data)}')
print('='*80)
display(tech_probs)

In [ ]:
# ============ 7. 置信区间计算 ============

def wilson_ci(p_hat, n, z=1.96):
    """Wilson得分区间"""
    denominator = 1 + z**2 / n
    center = (p_hat + z**2 / (2*n)) / denominator
    margin = z * np.sqrt((p_hat*(1-p_hat)/n + z**2/(4*n**2))) / denominator
    return max(0, center - margin), min(1, center + margin)

def calc_probs_with_ci(group_data, categories_order):
    """计算概率及Wilson置信区间"""
    total = len(group_data)
    results = []
    for cat in categories_order:
        count = (group_data['Category'] == cat).sum()
        prob = count / total if total > 0 else 0
        if count > 0:
            ci_low, ci_high = wilson_ci(prob, total)
        else:
            ci_low, ci_high = 0, 0
        results.append({
            '涨跌幅区间': cat,
            '样本数': count,
            '概率': round(prob, 4),
            '置信区间下界': round(ci_low, 4),
            '置信区间上界': round(ci_high, 4)
        })
    return pd.DataFrame(results)

# 计算带置信区间的结果
main_results = calc_probs_with_ci(main_data, main_order)
tech_results = calc_probs_with_ci(tech_data, tech_order)

print('='*100)
print('主板个股涨停次交易日涨跌幅概率分布（含95%置信区间）')
print(f'总样本数：{len(main_data)}')
print('='*100)
display(main_results)

print('\n' + '='*100)
print('创业板/科创板个股涨停次交易日涨跌幅概率分布（含95%置信区间）')
print(f'总样本数：{len(tech_data)}')
print('='*100)
display(tech_results)

In [ ]:
# ============ 8. 数据可视化 ============

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['PingFang SC', 'SimHei', 'Heiti SC']
matplotlib.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# 主板
ax1 = axes[0]
cats_main = main_results['涨跌幅区间'].tolist()
probs_main = main_results['概率'].tolist()
colors_main = ['#2ecc71' if '涨幅' in c or '涨停' in c else '#e74c3c' if '跌幅' in c or '跌停' in c else '#f39c12' for c in cats_main]
bars1 = ax1.bar(range(len(cats_main)), probs_main, color=colors_main, edgecolor='white', linewidth=1.2)
ax1.set_xticks(range(len(cats_main)))
ax1.set_xticklabels(cats_main, rotation=45, ha='right', fontsize=10)
ax1.set_title('主板涨停次交易日涨跌幅概率分布', fontsize=14, fontweight='bold')
ax1.set_ylabel('概率', fontsize=12)
ax1.set_ylim(0, max(probs_main) * 1.2)
for bar, prob in zip(bars1, probs_main):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005, 
             f'{prob:.2%}', ha='center', va='bottom', fontsize=9)

# 创业板/科创板
ax2 = axes[1]
cats_tech = tech_results['涨跌幅区间'].tolist()
probs_tech = tech_results['概率'].tolist()
colors_tech = ['#2ecc71' if '涨幅' in c or '涨停' in c else '#e74c3c' if '跌幅' in c or '跌停' in c else '#f39c12' for c in cats_tech]
bars2 = ax2.bar(range(len(cats_tech)), probs_tech, color=colors_tech, edgecolor='white', linewidth=1.2)
ax2.set_xticks(range(len(cats_tech)))
ax2.set_xticklabels(cats_tech, rotation=45, ha='right', fontsize=10)
ax2.set_title('创业板/科创板涨停次交易日涨跌幅概率分布', fontsize=14, fontweight='bold')
ax2.set_ylabel('概率', fontsize=12)
ax2.set_ylim(0, max(probs_tech) * 1.2)
for bar, prob in zip(bars2, probs_tech):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005, 
             f'{prob:.2%}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('Q1_涨停概率分布.png', dpi=150, bbox_inches='tight')
plt.show()

print('图表已保存为 Q1_涨停概率分布.png')

In [ ]:
# ============ 9. 统计摘要 ============

print('='*60)
print('统计摘要')
print('='*60)

# 主板统计
main_returns = main_data['Next_ChangeRatio'].dropna()
print(f'\n主板涨停后次交易日：')
print(f'  样本数：{len(main_returns)}')
print(f'  平均涨跌幅：{main_returns.mean()*100:.2f}%')
print(f'  中位数涨跌幅：{main_returns.median()*100:.2f}%')
print(f'  标准差：{main_returns.std()*100:.2f}%')
print(f'  最大涨幅：{main_returns.max()*100:.2f}%')
print(f'  最大跌幅：{main_returns.min()*100:.2f}%')
print(f'  上涨概率（次日收盘上涨）：{(main_returns > 0).mean()*100:.2f}%')

# 创业板/科创板统计
tech_returns = tech_data['Next_ChangeRatio'].dropna()
print(f'\n创业板/科创板涨停后次交易日：')
print(f'  样本数：{len(tech_returns)}')
print(f'  平均涨跌幅：{tech_returns.mean()*100:.2f}%')
print(f'  中位数涨跌幅：{tech_returns.median()*100:.2f}%')
print(f'  标准差：{tech_returns.std()*100:.2f}%')
print(f'  最大涨幅：{tech_returns.max()*100:.2f}%')
print(f'  最大跌幅：{tech_returns.min()*100:.2f}%')
print(f'  上涨概率（次日收盘上涨）：{(tech_returns > 0).mean()*100:.2f}%')